# Sprint 2 Colab Pipeline

This notebook is the primary Google Colab runner for heavy Sprint 2 execution. It is intentionally orchestration-only: raw-data staging, direct calls into the existing repository scripts, archive packaging, and optional download or private Google Drive copy.

It must not become the home of processing logic. The source of truth remains the Sprint 2 CLI scripts under `scripts/`.


## 1. Environment Setup

This section prepares the fresh Colab runtime. It only installs operational prerequisites needed to run the repository and create `.tar.zst` archives.


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")
print(f"Working directory before clone: {Path.cwd()}")

if shutil.which("zstd") is None:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
else:
    print("zstd is already available.")


## 2. Repository Setup

Edit the repository reference if you need a different branch, tag, or commit. The defaults target the public repository and the `master` branch.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "master"
WORKTREE_ROOT = Path("/content/text-to-sign-production")
RUNTIME_ROOT = WORKTREE_ROOT.parent

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
try:
    current_cwd = Path.cwd()
except FileNotFoundError:
    current_cwd = None

if current_cwd is None or current_cwd == WORKTREE_ROOT or WORKTREE_ROOT in current_cwd.parents:
    os.chdir(RUNTIME_ROOT)
    print(f"Changed working directory to {RUNTIME_ROOT} before refreshing the repo clone.")

if WORKTREE_ROOT.exists():
    shutil.rmtree(WORKTREE_ROOT)

subprocess.run(["git", "clone", REPO_URL, str(WORKTREE_ROOT)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=WORKTREE_ROOT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], cwd=WORKTREE_ROOT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], cwd=WORKTREE_ROOT, check=True)
os.chdir(WORKTREE_ROOT)

current_revision = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=WORKTREE_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Repository ready at {WORKTREE_ROOT}")
print(f"Checked out revision: {current_revision}")


## 3. Raw Data Fetch And Setup

Choose one raw-input mode:

- `public_urls`: download translation TSVs plus split `.tar.gz` keypoint archives from public How2Sign source URLs that you provide in this notebook.
- `mounted_paths`: copy already available raw inputs from user-controlled Colab or Google Drive paths.

This section resets the canonical raw staging directories inside the Colab clone before staging data into the Sprint 2 layout. In `public_urls` mode, extraction progress stays visible in the notebook while the extracted split is moved into place:

- `data/raw/how2sign/translations/`
- `data/raw/how2sign/bfh_keypoints/`


In [ ]:
from pathlib import Path

PROJECT_ROOT = WORKTREE_ROOT
RAW_ROOT = PROJECT_ROOT / "data/raw/how2sign"
TRANSLATIONS_DIR = RAW_ROOT / "translations"
BFH_KEYPOINTS_ROOT = RAW_ROOT / "bfh_keypoints"
DOWNLOAD_ROOT = Path("/content/how2sign_downloads")

RAW_INPUT_MODE = "mounted_paths"  # Change to "public_urls" when downloading from public sources.
PIPELINE_SPLITS = ["train", "val", "test"]  # Edit this list for subset runs.

TRANSLATION_URLS = {
    "train": "",
    "val": "",
    "test": "",
}
KEYPOINT_ARCHIVE_URLS = {
    "train": "",
    "val": "",
    "test": "",
}

MOUNTED_TRANSLATION_FILES = {
    "train": "",
    "val": "",
    "test": "",
}
MOUNTED_KEYPOINT_SPLIT_DIRS = {
    "train": "",
    "val": "",
    "test": "",
}

SPLIT_DIR_NAMES = {
    "train": "train_2D_keypoints",
    "val": "val_2D_keypoints",
    "test": "test_2D_keypoints",
}
TRANSLATION_FILE_NAMES = {
    "train": "how2sign_realigned_train.csv",
    "val": "how2sign_realigned_val.csv",
    "test": "how2sign_realigned_test.csv",
}


In [ ]:
import shutil
import subprocess
import urllib.request
from pathlib import Path
from urllib.parse import urlparse

import gdown


def _reset_directory(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def _require_values(mapping: dict[str, str], *, label: str, splits: list[str]) -> None:
    missing = [split for split in splits if not mapping.get(split)]
    if missing:
        raise ValueError(f"{label} is missing values for: {', '.join(missing)}")


def _is_google_drive_url(url: str) -> bool:
    host = urlparse(url).netloc.lower()
    return host in {"drive.google.com", "docs.google.com", "drive.usercontent.google.com"}


def _format_size(num_bytes: int) -> str:
    size = float(num_bytes)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if size < 1024 or unit == "TiB":
            if unit == "B":
                return f"{int(size)} {unit}"
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} TiB"


def _print_progress(prefix: str, completed: int, total: int, *, width: int = 24) -> None:
    if total <= 0:
        print(f"\r{prefix}: {_format_size(completed)} transferred", end="", flush=True)
        return
    ratio = min(completed / total, 1.0)
    filled = int(width * ratio)
    bar = "#" * filled + "-" * (width - filled)
    print(
        f"\r{prefix}: [{bar}] {ratio * 100:5.1f}% ({_format_size(completed)} / {_format_size(total)})",
        end="",
        flush=True,
    )


def _download_file(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if _is_google_drive_url(url):
        result = gdown.download(url=url, output=str(destination), quiet=False, fuzzy=True)
        if result is None:
            raise RuntimeError(f"Failed to download {url}")
    else:
        urllib.request.urlretrieve(url, destination)


def _close_process_stdin(process: subprocess.Popen) -> None:
    if process.stdin is None:
        return
    try:
        process.stdin.close()
    except BrokenPipeError:
        pass


def _stop_process(process: subprocess.Popen) -> int:
    if process.poll() is None:
        process.terminate()
        try:
            return process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
    return process.wait()


def _extract_tar_gz_with_progress(archive_path: Path, destination: Path) -> None:
    if not archive_path.name.endswith(".tar.gz"):
        raise ValueError(f"Expected a .tar.gz archive, got: {archive_path.name}")

    destination.mkdir(parents=True, exist_ok=True)
    total_bytes = archive_path.stat().st_size
    process = subprocess.Popen(
        ["tar", "-xzf", "-", "-C", str(destination)],
        stdin=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    try:
        with archive_path.open("rb") as archive_stream:
            while chunk := archive_stream.read(8 * 1024 * 1024):
                if process.stdin is None:
                    raise RuntimeError(f"tar stdin was not available for {archive_path}")
                process.stdin.write(chunk)
                completed = min(archive_stream.tell(), total_bytes)
                _print_progress("Extraction progress", completed, total_bytes)
    except BrokenPipeError as exc:
        _close_process_stdin(process)
        returncode = _stop_process(process)
        print()
        raise RuntimeError(
            f"tar extraction failed for {archive_path} with exit code {returncode}. See notebook output above for details."
        ) from exc
    except BaseException:
        _close_process_stdin(process)
        _stop_process(process)
        print()
        raise

    _close_process_stdin(process)
    returncode = process.wait()
    print()
    if returncode != 0:
        raise RuntimeError(
            f"tar extraction failed for {archive_path} with exit code {returncode}. See notebook output above for details."
        )


def _contains_openpose_json(path: Path) -> bool:
    return (path / "openpose_output" / "json").is_dir()


def _find_split_dir(search_root: Path, expected_dir_name: str) -> Path:
    expected_path = search_root / expected_dir_name
    if _contains_openpose_json(expected_path):
        return expected_path
    if _contains_openpose_json(search_root):
        return search_root

    expected_candidates = sorted(
        path
        for path in search_root.rglob(expected_dir_name)
        if path.is_dir() and _contains_openpose_json(path)
    )
    if len(expected_candidates) == 1:
        return expected_candidates[0]
    if len(expected_candidates) > 1:
        raise RuntimeError(
            f"Ambiguous extracted split directories under {search_root}: {expected_candidates}"
        )

    derived_candidates = sorted(
        {
            path.parent
            for path in search_root.rglob("openpose_output")
            if path.is_dir() and (path / "json").is_dir()
        },
        key=lambda path: str(path),
    )
    if not derived_candidates:
        raise RuntimeError(
            f"Could not locate an extracted split directory with openpose_output/json under {search_root}."
        )
    if len(derived_candidates) > 1:
        raise RuntimeError(
            f"Ambiguous extracted split directories under {search_root}: {derived_candidates}"
        )
    return derived_candidates[0]


def _copy_tree(source: Path, destination: Path) -> None:
    shutil.copytree(source, destination, dirs_exist_ok=True)


def _stage_public_split(split: str) -> None:
    translation_url = TRANSLATION_URLS[split]
    translation_destination = TRANSLATIONS_DIR / TRANSLATION_FILE_NAMES[split]
    archive_url = KEYPOINT_ARCHIVE_URLS[split]
    archive_path = DOWNLOAD_ROOT / f"{split}_keypoints_archive.tar.gz"
    extract_root = DOWNLOAD_ROOT / f"{split}_extract"
    canonical_destination = BFH_KEYPOINTS_ROOT / SPLIT_DIR_NAMES[split]

    if archive_path.exists():
        archive_path.unlink()
    if extract_root.exists():
        shutil.rmtree(extract_root)
    if canonical_destination.exists():
        shutil.rmtree(canonical_destination)

    try:
        print(f"[{split}] translation download starting")
        _download_file(translation_url, translation_destination)
        print(f"[{split}] translation download finished")

        print(f"[{split}] keypoint archive download starting")
        _download_file(archive_url, archive_path)
        print(f"[{split}] keypoint archive download finished")

        print(f"[{split}] extraction starting")
        _extract_tar_gz_with_progress(archive_path, extract_root)
        print(f"[{split}] extraction finished")

        print(f"[{split}] locating extracted split directory")
        split_dir = _find_split_dir(extract_root, SPLIT_DIR_NAMES[split])

        print(f"[{split}] moving into canonical layout")
        shutil.move(str(split_dir), str(canonical_destination))
    finally:
        if archive_path.exists() or extract_root.exists():
            print(f"[{split}] cleaning temporary files")
        archive_path.unlink(missing_ok=True)
        if extract_root.exists():
            shutil.rmtree(extract_root)


def _stage_mounted_split(split: str) -> None:
    translation_source = Path(MOUNTED_TRANSLATION_FILES[split])
    translation_destination = TRANSLATIONS_DIR / TRANSLATION_FILE_NAMES[split]
    keypoint_source = Path(MOUNTED_KEYPOINT_SPLIT_DIRS[split])
    keypoint_destination = BFH_KEYPOINTS_ROOT / SPLIT_DIR_NAMES[split]

    print(f"[{split}] copying mounted translation into canonical layout")
    translation_destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(translation_source, translation_destination)

    print(f"[{split}] copying mounted keypoints into canonical layout")
    _copy_tree(keypoint_source, keypoint_destination)


_reset_directory(TRANSLATIONS_DIR)
_reset_directory(BFH_KEYPOINTS_ROOT)
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Selected splits: {PIPELINE_SPLITS}")

if RAW_INPUT_MODE == "public_urls":
    _require_values(TRANSLATION_URLS, label="TRANSLATION_URLS", splits=PIPELINE_SPLITS)
    _require_values(KEYPOINT_ARCHIVE_URLS, label="KEYPOINT_ARCHIVE_URLS", splits=PIPELINE_SPLITS)

    for split in PIPELINE_SPLITS:
        _stage_public_split(split)
elif RAW_INPUT_MODE == "mounted_paths":
    _require_values(MOUNTED_TRANSLATION_FILES, label="MOUNTED_TRANSLATION_FILES", splits=PIPELINE_SPLITS)
    _require_values(MOUNTED_KEYPOINT_SPLIT_DIRS, label="MOUNTED_KEYPOINT_SPLIT_DIRS", splits=PIPELINE_SPLITS)

    for split in PIPELINE_SPLITS:
        _stage_mounted_split(split)
else:
    raise ValueError("RAW_INPUT_MODE must be either 'public_urls' or 'mounted_paths'.")

for split in PIPELINE_SPLITS:
    translation_path = TRANSLATIONS_DIR / TRANSLATION_FILE_NAMES[split]
    keypoints_dir = BFH_KEYPOINTS_ROOT / SPLIT_DIR_NAMES[split] / "openpose_output" / "json"
    if not translation_path.exists():
        raise FileNotFoundError(f"Missing translation file: {translation_path}")
    if not keypoints_dir.is_dir():
        raise FileNotFoundError(f"Missing keypoint directory: {keypoints_dir}")
    print(f"[{split}] split ready")

print(f"Canonical raw layout staged under {RAW_ROOT}")


## 4. Sprint 2 Script Execution

Colab heavy runs call the existing scripts directly instead of forcing `dvc repro`. DVC remains the repository standard for reproducibility, but direct script execution avoids unnecessary hashing overhead over very large raw trees in the Colab runtime.


In [ ]:
import shlex
import subprocess
import sys

commands = [
    [sys.executable, "scripts/prepare_raw.py", "--splits", *PIPELINE_SPLITS],
    [sys.executable, "scripts/normalize_keypoints.py", "--splits", *PIPELINE_SPLITS],
    [sys.executable, "scripts/filter_samples.py", "--config", "configs/data/filter-v1.yaml", "--splits", *PIPELINE_SPLITS],
    [sys.executable, "scripts/export_training_manifest.py", "--splits", *PIPELINE_SPLITS],
]

for command in commands:
    print(f"$ {shlex.join(command)}")
    subprocess.run(command, cwd=WORKTREE_ROOT, check=True)

print("Sprint 2 pipeline completed.")


## 5. Output Packaging

This section loads optional storage settings, then creates explicit `.tar.zst` archives for manifests, reports, and per-split processed samples.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import subprocess
import sys
import yaml

storage_config_path = WORKTREE_ROOT / "configs/storage.local.yaml"
storage_example_path = WORKTREE_ROOT / "configs/storage.example.yaml"

if storage_config_path.exists():
    storage_config = yaml.safe_load(storage_config_path.read_text(encoding="utf-8"))
    print("Loaded local storage config override.")
else:
    storage_config = yaml.safe_load(storage_example_path.read_text(encoding="utf-8"))
    print("Loaded example storage config.")

if storage_config.get("run_label") == "sprint2-colab-example-run":
    storage_config["run_label"] = datetime.now(timezone.utc).strftime("sprint2-colab-%Y%m%dT%H%M%SZ")

subprocess.run(
    [sys.executable, "scripts/package_sprint2_outputs.py", "--splits", *PIPELINE_SPLITS],
    cwd=WORKTREE_ROOT,
    check=True,
)

archive_root = WORKTREE_ROOT / "data/archives"
archive_paths = [archive_root / "sprint2_manifests_reports.tar.zst"] + [
    archive_root / f"sprint2_samples_{split}.tar.zst" for split in PIPELINE_SPLITS
]
for archive_path in archive_paths:
    if not archive_path.exists():
        continue
    size_mib = archive_path.stat().st_size / (1024 * 1024)
    print(f"{archive_path.name}: {size_mib:.2f} MiB")

print(
    {
        "provider": storage_config.get("provider"),
        "run_label": storage_config.get("run_label"),
        "local_download": storage_config.get("local_download"),
        "upload_to_storage": storage_config.get("upload_to_storage"),
    }
)


## 6. Download Or Private Storage Copy

The repository never commits these large outputs to GitHub. Use browser downloads for smaller archives when practical, or mount Google Drive and copy archives to a private/shared location that is not published through GitHub Pages.


In [ ]:
import shutil
from pathlib import Path

DOWNLOAD_ARCHIVE_NAMES = ["sprint2_manifests_reports.tar.zst"]
UPLOAD_ARCHIVE_NAMES = [archive_path.name for archive_path in archive_paths]

if storage_config.get("local_download", False):
    try:
        from google.colab import files
    except ModuleNotFoundError:
        print("google.colab.files is unavailable outside Colab.")
    else:
        for archive_name in DOWNLOAD_ARCHIVE_NAMES:
            archive_path = archive_root / archive_name
            if not archive_path.exists():
                raise FileNotFoundError(f"Archive not found for download: {archive_path}")
            print(f"Downloading {archive_name}")
            files.download(str(archive_path))
else:
    print("local_download is disabled in storage config.")

if storage_config.get("upload_to_storage", False):
    drive_config = storage_config.get("google_drive", {})
    mount_path = Path(drive_config.get("mount_path", "/content/drive"))
    relative_root = Path(drive_config.get("relative_root", "MyDrive/text-to-sign-production/artifacts"))
    artifact_root = Path(storage_config.get("artifact_root", "text-to-sign-production"))
    run_label = storage_config["run_label"]

    try:
        from google.colab import drive
    except ModuleNotFoundError:
        print("google.colab.drive is unavailable outside Colab.")
    else:
        drive.mount(str(mount_path), force_remount=False)
        target_root = mount_path / relative_root / artifact_root / run_label
        target_root.mkdir(parents=True, exist_ok=True)
        for archive_name in UPLOAD_ARCHIVE_NAMES:
            archive_path = archive_root / archive_name
            destination = target_root / archive_name
            shutil.copy2(archive_path, destination)
            print(f"Copied {archive_name} -> {destination}")
else:
    print("upload_to_storage is disabled in storage config.")
